# BASELINE CLASSIFICATION: TF-IDF + LOGISTIC REGRESSION / LINEAR SVC 
**Purpose**: Establish baseline performance for multi-label classification

**Models**: 
1. TF-IDF + Logistic Regression (OneVsRest)
2. TF-IDF + Linear SVC (OneVsRest) 

**Output**: 
- Binary label predictions for comparison with BERT methods

In [2]:
# Imports                 

import pandas as pd
import numpy as np
import re
import json
from pathlib import Path
from collections import Counter
from itertools import combinations

from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.multiclass import OneVsRestClassifier

from su_utils import join_uo_by_course_version, primary_from_labels, build_text, clean_text,
serialize_tuple, eval_multilabel

from joblib import dump
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
print('Imports ok.')

Imports ok.


### 1. CONFIGURATION & PATHS

In [3]:

DATA_DIR = Path('/kaggle/input/course-descriptions-2023-raw')
CSV_PATH = DATA_DIR / 'Kursplanekorpus-2023-original-ej-bearb.csv'

ARTIFACTS_DIR = Path('/kaggle/working/artifacts')
ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)

MODEL_DIR = Path('/kaggle/working/tfidf_baseline')
MODEL_DIR.mkdir(exist_ok=True, parents=True)

EXPORTS_DIR = Path('/kaggle/working/exports')
EXPORTS_DIR.mkdir(exist_ok=True, parents=True)

RANDOM_SEED = 42
print('Paths ok.')

# Column definitions
ID_COL    = "id"
UO_COL    = "education_area_list_id"
PCT_COL   = "percentage"
TEXT_COLS = ["name", "plan_description", "objectives"]
DATE_COLS = ["decision_date", "start_date"]
print("Configuration complete.")

Paths ok.
Configuration complete.



### 2. DATA LOADING & PREPROCESSING

**2.1 Load raw data**

In [4]:
df_raw = pd.read_csv(CSV_PATH, sep=";", low_memory=False)
print(f"Raw data loaded: {df_raw.shape[0]} rows")

Raw data loaded: 12948 rows


**2.2 Rename and convert columns**

In [6]:
df = df_raw.rename(columns={
    "education_area_list_id": "uo_code",
    "percentage": "uo_percentage",
})

# Convert date columns
for date_col in DATE_COLS:
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')

# Parse UO codes and percentages
df["uo_code"] = pd.to_numeric(df["uo_code"], errors="coerce").astype("Int64")
df["uo_percentage"] = (
    df["uo_percentage"].astype(str).str.replace(",", ".", regex=False)
               .pipe(pd.to_numeric, errors="coerce")
)
print("Columns renamed")

Columns renamed


**2.3 Join multi-row UO information**

In [7]:
df_joined = join_uo_by_course_version(df)
df = df_joined
print(f"After joining UO rows: {df.shape[0]} rows")

After joining UO rows: 9770 rows


**2.4 Define primary UO for stratified sampling**

In [12]:
df["primary_uo"] = df.apply(
    lambda r: primary_from_labels(r["labels_uo"], r["labels_pct"]),
    axis=1
)
df = df.dropna(subset=["primary_uo"])
df["primary_uo"] = df["primary_uo"].astype(int)
print(f"After setting primary UO: {df.shape[0]} rows")



After setting primary UO: 9770 rows


**2.5 Drop null columns and build text**
We concatenate relevant text columns ("name", "plan_description", "objectives") into one column (all columns are kept). Then we perform some basic cleaning

In [9]:
df = df.dropna(axis=1)  # Drop columns with missing values
df["text"] = df.apply(build_text, axis=1, args=(TEXT_COLS,))
df["text"] = df["text"].apply(clean_text)
df = df[df["text"].str.len() > 10]  # Remove very short texts
print(f"Final preprocessed data: {df.shape[0]} rows, {df.shape[1]} columns")

Final preprocessed data: 9770 rows, 27 columns


**2.6 Display label distribution**

In [13]:
# Display label distribution
print("\nPrimary UO distribution:")
print(df["primary_uo"].value_counts().sort_index())


Primary UO distribution:
primary_uo
2434    3332
2436     452
2438     317
2439     327
2441    1691
2442    2234
2444     457
2445     157
2447     684
2451     119
Name: count, dtype: int64


### 3. TRAIN/VAL SPLIT (STRATIFIED BY PRIMARY UO)
We want to split our dataset into train, test and validation set, but first we want to make sure that these subsets will be representative of the original superset (i e we want proportional splits)

**3.1 Build course-level dataframe for splitting**

In [14]:
courses = (
    df[["id", "primary_uo", "decision_date"]]
      .sort_values(["id", "decision_date"])
      .groupby("id", as_index=False)
      .last()
)
print(f"Unique courses: {len(courses)}")

Unique courses: 4880


In [ ]:
# Legacy: from earlier version, will delete
## We find that we have 25 unique label combinations:
## Convert tuple to frozenset so sets with same members are treated as identical combinations
#df["label_set"] = df["labels_uo"].apply(lambda t: frozenset(t))

## Count unique combinations
#unique_sets = df["label_set"].value_counts()

#print(unique_sets)
#print("Number of unique label combinations:", unique_sets.shape[0])


**3.2 Stratified split**

In [15]:
# Perform stratified split into train and val sets based on primary_uo

train_ids, val_ids = train_test_split(
    courses["id"],
    test_size=0.2,
    random_state=42,
    stratify=courses["primary_uo"].astype(str),
)

print("Train ids:", len(train_ids), "Val ids:", len(val_ids))


Train ids: 3904 Val ids: 976


**3.3 Map back to full dataframe (includes all versions)**

In [16]:
# We map back to full multilabel df by filtering on id
train_df = df[df["id"].isin(train_ids)].copy()
val_df   = df[df["id"].isin(val_ids)].copy()

print("Train rows:", train_df.shape, "Val rows:", val_df.shape)
print("Train ids unique:", train_df["id"].nunique())
print("Val ids unique:",   val_df["id"].nunique())
assert set(train_df["id"]).isdisjoint(set(val_df["id"]))

Train rows: (7865, 27) Val rows: (1905, 27)
Train ids unique: 3904
Val ids unique: 976


### 4. PREPARE MULTILABEL DATA

In [17]:
LABEL_COL = "labels_uo"
PCT_COL = "labels_pct"

def make_ml_df(df_split):
    """Extract relevant columns for multilabel learning."""
    ml = df_split[["id", "text", LABEL_COL, PCT_COL]].copy()
    ml["text"] = ml["text"].astype("string").fillna("")
    ml = ml.dropna(subset=["text", LABEL_COL])
    ml = ml[ml["text"].str.len() > 10]
    return ml

train_ml_df = make_ml_df(train_df)
val_ml_df   = make_ml_df(val_df)
print(f"Train ML: {len(train_ml_df)} samples")
print(f"Val ML:   {len(val_ml_df)} samples")

Train ML: 7865 samples
Val ML:   1905 samples


### 5. MULTILABEL BINARIZATION

In [18]:
y_train = train_ml_df["labels_uo"].tolist()
y_val   = val_ml_df["labels_uo"].tolist()

mlb = MultiLabelBinarizer()
Y_train = mlb.fit_transform(y_train)
Y_val   = mlb.transform(y_val)

# Convert to float32 for consistency with BERT pipeline
Y_train_float = Y_train.astype("float32")
Y_val_float   = Y_val.astype("float32")

label_list = mlb.classes_
num_labels = len(label_list)

print(f"Number of labels: {num_labels}")
print(f"UO codes: {label_list}")
print(f"Train label matrix: {Y_train.shape}")
print(f"Val label matrix:   {Y_val.shape}")

# Label distribution
label_counts = Y_train.sum(axis=0)
print("\nLabel frequencies (train set):")
for uo, count in zip(label_list, label_counts):
    print(f"  {uo}: {count:4d} ({count/len(Y_train)*100:.1f}%)")

Number of labels: 10
UO codes: [2434 2436 2438 2439 2441 2442 2444 2445 2447 2451]
Train label matrix: (7865, 10)
Val label matrix:   (1905, 10)

Label frequencies (train set):
  2434: 2721 (34.6%)
  2436:  375 (4.8%)
  2438:  356 (4.5%)
  2439:  380 (4.8%)
  2441: 1484 (18.9%)
  2442: 2664 (33.9%)
  2444:  390 (5.0%)
  2445:  288 (3.7%)
  2447:  607 (7.7%)
  2451:  151 (1.9%)


### 6. BASELINE MODEL 1: TF-IDF + LOGISTIC REGRESSION
We train our first baseline model.

In [19]:
logreg_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3,5),
        min_df=3,
        max_df=0.9
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(max_iter=1000, class_weight="balanced")
    ))
])

logreg_pipe.fit(train_ml_df["text"], Y_train)
Y_prob = logreg_pipe.predict_proba(val_ml_df["text"])
Y_pred_log = (Y_prob >= 0.5).astype(int)
print("Predictions complete.")

Predictions complete.


###  7. BASELINE MODEL 2: TF-IDF + LINEAR SVC
We train our second baseline model.

In [21]:
svc_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3,5),
        min_df=3,
        max_df=0.9
    )),
    ("clf", OneVsRestClassifier(LinearSVC()))
])


print("Fitting model...")
svc_pipe.fit(train_ml_df["text"], Y_train)
print("✓ Model trained")

# Predict (SVC returns binary, not probabilities)
Y_pred_svc = svc_pipe.predict(val_ml_df["text"])

print("Predictions complete.")

Fitting model...
✓ Model trained
Predictions complete.


In [23]:
# Evaluate both models
logreg_metrics = eval_multilabel(Y_val, Y_pred_log, label_list)
svc_metrics = eval_multilabel(Y_val, Y_pred_svc, label_list)


print("LogReg metrics:", logreg_metrics)
print("LinearSVC metrics:", svc_metrics)

LogReg metrics: ({'subset_accuracy': 0.8136482939632546, 'micro_f1': 0.8967836871978138, 'macro_f1': 0.8444800057691262, 'hamming_loss': 0.025774278215223097},    label        f1
0   2436  0.949367
1   2434  0.940919
2   2441  0.930748
3   2442  0.919003
4   2439  0.854460
5   2445  0.845771
6   2447  0.830189
7   2438  0.735178
8   2444  0.734940
9   2451  0.704225)
LinearSVC metrics: ({'subset_accuracy': 0.8792650918635171, 'micro_f1': 0.9277703005832212, 'macro_f1': 0.8926912835783638, 'hamming_loss': 0.016902887139107613},    label        f1
0   2441  0.966715
1   2436  0.960526
2   2434  0.956650
3   2442  0.932794
4   2451  0.877193
5   2447  0.868613
6   2439  0.861702
7   2445  0.844444
8   2438  0.839378
9   2444  0.818898)


### 8. EXPORT DATA FOR BERT PIPELINE

****

**8.1 Serialize tuple columns and binarized label columns**

In [25]:
# Build export dataframes
train_ml_export = train_ml_df[["id", "text", "labels_uo", "labels_pct"]].copy()
val_ml_export   = val_ml_df[["id", "text", "labels_uo", "labels_pct"]].copy()
print('')

# Serialize tuple columns
train_ml_export["labels_uo"]  = train_ml_export["labels_uo"].apply(serialize_tuple)
train_ml_export["labels_pct"] = train_ml_export["labels_pct"].apply(serialize_tuple)
val_ml_export["labels_uo"]    = val_ml_export["labels_uo"].apply(serialize_tuple)
val_ml_export["labels_pct"]   = val_ml_export["labels_pct"].apply(serialize_tuple)
print("Dataframes built")
# Add binarized label columns
for i, lab in enumerate(label_list):
    col = f"y_{lab}"
    train_ml_export[col] = Y_train[:, i]
    val_ml_export[col]   = Y_val[:, i]
print("binarized columns added")


Dataframes built
binarized columns added


**8.2 Save Everything for export**

In [26]:
# Save exports
train_ml_export.to_csv(EXPORTS_DIR / "train_ml_export.csv", index=False)
val_ml_export.to_csv(EXPORTS_DIR / "val_ml_export.csv", index=False)

# Save label order
dump(label_list.tolist(), EXPORTS_DIR / "uo_label_list.joblib")

print(f"✓ Exported {len(train_ml_export)} train rows")
print(f"✓ Exported {len(val_ml_export)} val rows")
print(f"✓ Columns: {list(train_ml_export.columns)}")

✓ Exported 7865 train rows
✓ Exported 1905 val rows
✓ Columns: ['id', 'text', 'labels_uo', 'labels_pct', 'y_2434', 'y_2436', 'y_2438', 'y_2439', 'y_2441', 'y_2442', 'y_2444', 'y_2445', 'y_2447', 'y_2451']


In [39]:
#Save baseline models

# Save the Logistic Regression pipeline
dump(logreg_pipe, MODEL_DIR / "logreg_pipe.joblib")
print("saved logreg pipeline")
# Save the SVC pipeline
dump(svc_pipe, MODEL_DIR / "svc_pipe.joblib")
print("saved svc pipeline")

['/kaggle/working/tfidf_baseline/svc_pipe.joblib']

### 9. Summing up 
What was acheived:

In [37]:
metrics_dict = svc_metrics[0]

print("\n" + "="*80)
print("BASELINE TRAINING COMPLETE")
print("="*80)
print(f"\nBest performing baseline: Linear SVC")
print(f"  Subset Accuracy: {metrics_dict['subset_accuracy']:.4f}")
print(f"  Micro F1:        {metrics_dict['micro_f1']:.4f}")
print(f"  Macro F1:        {metrics_dict['macro_f1']:.4f}")
print(f"  Hamming Loss:    {metrics_dict['hamming_loss']:.4f}")
print(f"\nExported files:")
print(f"  - train_ml_export.csv ({len(train_ml_export)} rows)")
print(f"  - val_ml_export.csv ({len(val_ml_export)} rows)")
print(f"  - uo_label_list.joblib (label ordering)")
print("\nReady for BERT fine-tuning.")



BASELINE TRAINING COMPLETE

Best performing baseline: Linear SVC
  Subset Accuracy: 0.8793
  Micro F1:        0.9278
  Macro F1:        0.8927
  Hamming Loss:    0.0169

Exported files:
  - train_ml_export.csv (7865 rows)
  - val_ml_export.csv (1905 rows)
  - uo_label_list.joblib (label ordering)

Ready for BERT fine-tuning.
